In [1]:
from pathlib import Path
import pandas as pd
import re

In [2]:
data_path = Path("clean_data/programa_de_apoio_ao_interior_clean.csv")
df = pd.read_csv(data_path)
df.head()

,query,original_url,archive_url,title,archive_date,year,archive_date_dt,snippet_clean,location_name,location_source,source_file
0,programa de apoio ao interior,http://www.partido-socialista.pt/ps/programa/p...,https://arquivo.pt/wayback/19961013180604/http...,I - OS PORTUGUESES SABEM O QUE QUEREM,845229964,1996,8452-02-09 09:06:04,"social de emprego, bem como nas políticas acti...",políticas activas de emprego,snippet_clean,programa_de_apoio_ao_interior_1996_C.csv
1,programa de apoio ao interior,http://www.jnoticias.pt/angola1.htm,https://arquivo.pt/wayback/19961013180528/http...,Angola: 20 anos depois - 1,845229928,1996,8452-02-09 09:02:08,"ao golpe militar em Portugal, com proclamações...",Portugal,snippet_clean,programa_de_apoio_ao_interior_1996_C.csv
2,programa de apoio ao interior,http://amadeus.inesc.pt/~jota/RSF/rsf.9,https://arquivo.pt/wayback/19961013210902/http...,NaN,845240942,1996,NaN,Fim fin. ZIE ENDE. the end. Mais um episodio d...,proxima sexta,snippet_clean,programa_de_apoio_ao_interior_1996_C.csv
3,programa de apoio ao interior,http://pcserver2.uac.pt/CAH/Pangra.htm,https://arquivo.pt/wayback/19961102050400/http...,Campus de Angra do Heroísmo,846911040,1996,8469-01-01 00:04:00,sua disposição um conjunto serviços de apoio q...,Achada,snippet_clean,programa_de_apoio_ao_interior_1996_C.csv
4,programa de apoio ao interior,http://euroinfo.ce.pt/info/fax/fax7.html,https://arquivo.pt/wayback/19961031125322/http...,EIP - Outros Documentos/Informações,846766402,1996,8467-06-06 04:00:02,DE ALTERAÇÃO AO REGIME DE IMPORTAÇÃO TELECOMUN...,UNIÃO EUROPEIA ACÇÃO PÚBLICA JOVENS,snippet_clean,programa_de_apoio_ao_interior_1996_C.csv


## 1) Função de relevância para políticas públicas
Criámos um sistema simples e transparente baseado em palavras-chave para identificação (flagging) de conteúdo relevante. De seguida, verificamos manualmente os resultados obtidos, validando a eficácia do método e ajustando as palavras-chave conforme necessário.

In [3]:
policy_keywords = [
    "interior do país", "interior", "desenvolvimento", "investimento", "incentivos",
    "quadro comunitário de apoio", "qca", "programa do governo", "política",
    "regional", "fixação da população", "infra-estruturas", "coesão", "coesao",
    "emprego", "turismo", "agricultura", "indústria", "industria", "fundos"
 ]
non_policy_keywords = [
    "programa de", "disciplina", "manual", "tutorial", "curso", "teatro",
    "festival", "agenda", "exposição", "exposicao", "software", "tese"
 ]

text_source = (
    df.get("title", pd.Series(dtype="object")).fillna("").astype(str)
    + " "
    + df.get("snippet_clean", pd.Series(dtype="object")).fillna("").astype(str)
 )

def contains_any(text: str, keywords: list[str]) -> bool:
    t = text.lower()
    return any(k in t for k in keywords)

df["policy_hit"] = text_source.map(lambda t: contains_any(t, policy_keywords))
df["non_policy_hit"] = text_source.map(lambda t: contains_any(t, non_policy_keywords))
df["policy_relevant"] = df["policy_hit"] & ~df["non_policy_hit"]

policy_counts = df["policy_relevant"].value_counts(dropna=False)
policy_counts

policy_relevant
False    6095
True     3160
Name: count, dtype: int64

## 2) Resumos das políticas de apoio e identificação de domínios
Aplicamos o filtro de relevência política ao subconjunto, isolando os documentos verdadeiramente orientados por política pública. Sobre este subconjunto, extraímos e sumarizamos os sinais de política recorrentes, identificamos os domínios institucionais com maior presença narrativa e analisamos as tendências temporais, revelando como o discurso político sobre o interior evoluiu de 1996 até 2020.

In [4]:
from urllib.parse import urlparse

policy_df = df[df["policy_relevant"]].copy()

def get_domain(url: str) -> str:
    if not isinstance(url, str) or not url.strip():
        return ""
    return urlparse(url).netloc.lower()

policy_df["domain"] = policy_df.get("original_url", "").map(get_domain)

top_domains = policy_df["domain"].value_counts().head(15)
top_domains

domain
www.ointerior.pt          13
www.apbad.pt              11
www.publico.pt            10
www.min-agricultura.pt     9
www.freipedro.pt           9
www.portugal.gov.pt        9
www.pcm.gov.pt             8
www.pcp.pt                 8
www.parlamento.pt          8
www.amal.pt                8
www.iefp.pt                8
www.regiao-sul.pt          8
www.ensino.eu              8
www.cidadevirtual.pt       7
www.dge.ubi.pt             7
Name: count, dtype: int64

In [5]:
theme_keywords = {
    "funding_qca_eu": ["quadro comunitário de apoio", "qca", "fundos", "feder"],
    "employment": ["emprego", "desemprego", "formação", "formacao"],
    "infrastructure": ["infra-estruturas", "infraestruturas", "rede", "transportes"],
    "regional_dev": ["desenvolvimento", "coesão", "coesao", "interior"],
    "investment_business": ["investimento", "incentivos", "empresas", "pme"],
    "agri_rural": ["agricultura", "rural", "aldeias"],
    "tourism_culture": ["turismo", "cultura", "património", "patrimonio"],
}

policy_text = (
    policy_df.get("title", "").fillna("").astype(str)
    + " "
    + policy_df.get("snippet_clean", "").fillna("").astype(str)
 ).str.lower()

theme_counts = {
    theme: policy_text.str.contains("|".join(map(re.escape, terms)), regex=True).sum()
    for theme, terms in theme_keywords.items()
}

pd.Series(theme_counts).sort_values(ascending=False)

regional_dev           2595
investment_business     343
tourism_culture         341
employment              339
infrastructure          187
funding_qca_eu          150
agri_rural              133
dtype: int64

In [6]:
year_series = pd.to_numeric(policy_df.get("year", pd.Series(dtype="object")), errors="coerce")
year_counts = (
    policy_df.assign(year_num=year_series)
    .dropna(subset=["year_num"])
    .groupby("year_num").size()
    .sort_index()
 )

year_counts

year_num
1996      2
1997     21
1998     40
1999     59
2000     87
2001    111
2002    102
2003    131
2004    113
2005    153
2006    147
2007     56
2008    150
2009    144
2010    187
2011    165
2012    173
2013    183
2014    153
2015    164
2016    135
2017    161
2018    173
2019    184
2020    166
dtype: int64

## 3) Narrativas dos Media
Focámo-nos nos domínios mediáticos para perceber como a narrativa da política para o interior é enquadrada e as frequências associadas.


In [8]:
if "domain" not in policy_df.columns:
    policy_df["domain"] = policy_df.get("original_url", "").map(get_domain)

media_domains = [
    "publico.pt", "expresso.pt", "dn.pt", "lusa.pt", "jnoticias.pt", "portugal-linha.pt",
    "tribunapress.pt", "voz-portucalense.pt", "localnet.pt", "netitude.pt", "domdigital.pt",
    "sapo.pt", "a-pagina-da-educacao.pt", "ecclesia.pt"
 ]

media_mask = policy_df["domain"].str.contains("|".join(map(re.escape, media_domains)), na=False)
media_df = policy_df[media_mask].copy()

media_by_domain = media_df["domain"].value_counts().head(15)
media_by_domain

domain
www.publico.pt                        10
www.jnoticias.pt                       5
jornalpraceta.no.sapo.pt               5
www.agencia.ecclesia.pt                5
bill.publico.pt                        4
dossiers.publico.pt                    4
dn.sapo.pt                             4
industriadarte.blogs.sapo.pt           4
luisnorbertolourenco.blogs.sapo.pt     4
turismo-beja.blogs.sapo.pt             4
cidadeagar.blogs.sapo.pt               4
agencia.ecclesia.pt                    4
voz-portucalense.pt                    3
www.expresso.pt                        3
abemdanacao.blogs.sapo.pt              3
Name: count, dtype: int64

In [9]:
media_year = pd.to_numeric(media_df.get("year", pd.Series(dtype="object")), errors="coerce")
media_year_counts = (
    media_df.assign(year_num=media_year)
    .dropna(subset=["year_num"])
    .groupby("year_num").size()
    .sort_index()
 )

media_year_counts

year_num
1996     1
1997     3
1998     3
1999     8
2000     7
2001     5
2002     6
2003     9
2004     9
2005    21
2006    19
2007    31
2008    30
2009    13
2010    81
2011    16
2012     8
2013    16
2014    56
2015    16
2016    11
2017     3
2018    11
2019    16
2020     8
dtype: int64

In [10]:
media_text = (
    media_df.get("title", "").fillna("").astype(str)
    + " "
    + media_df.get("snippet_clean", "").fillna("").astype(str)
 ).str.lower()

media_theme_counts = {
    theme: media_text.str.contains("|".join(map(re.escape, terms)), regex=True).sum()
    for theme, terms in theme_keywords.items()
}

pd.Series(media_theme_counts).sort_values(ascending=False)

regional_dev           342
investment_business     39
tourism_culture         39
employment              25
infrastructure          11
agri_rural              10
funding_qca_eu           8
dtype: int64

## 4) Tese de desenvolvimento regional
Isolamos a linguagem mais explicitamente orientada para o desenvolvimento regional para apoiar as narrativas relacionadas com o interior e infraestruturas.

In [11]:
dev_keywords = [
    "interior", "desenvolvimento", "coesão", "coesao", "regional", "infra-estruturas",
    "infraestruturas", "investimento", "incentivos", "fixação da população", "emprego"
 ]

dev_mask = policy_text.str.contains("|".join(map(re.escape, dev_keywords)), regex=True)
dev_df = policy_df[dev_mask].copy()

dev_by_domain = dev_df["domain"].value_counts().head(15)
dev_by_domain

domain
www.ointerior.pt           13
www.apbad.pt               11
www.min-agricultura.pt      9
www.freipedro.pt            9
www.pcp.pt                  8
www.amal.pt                 8
www.publico.pt              8
www.iefp.pt                 8
www.portugal.gov.pt         8
www.ensino.eu               8
www.pcm.gov.pt              7
www.parlamento.pt           7
www.cidadevirtual.pt        7
www.dge.ubi.pt              7
www.noticiasdeaveiro.pt     7
Name: count, dtype: int64

In [12]:
dev_year = pd.to_numeric(dev_df.get("year", pd.Series(dtype="object")), errors="coerce")
dev_year_counts = (
    dev_df.assign(year_num=dev_year)
    .dropna(subset=["year_num"])
    .groupby("year_num").size()
    .sort_index()
 )

dev_year_counts

year_num
1996      2
1997     19
1998     39
1999     57
2000     83
2001     97
2002     94
2003    116
2004    105
2005    138
2006    135
2007     49
2008    138
2009    126
2010    174
2011    147
2012    149
2013    168
2014    134
2015    149
2016    116
2017    153
2018    156
2019    173
2020    154
dtype: int64